## Preprocessing Data

In [1]:
import mne
import numpy as np
import polars as pl

In [2]:
DEF_PATH = "Bonn-EEG-Dataset-main/"
sets = ["F", "N", "O", "S", "Z"]
sets = ["O", "N", "S"]
sets_index_map ={ 
    "F": 1,
    "N": 2,
    "O": 3,
    "S": 4,
    "Z": 5
}
rows = []

for s in sets:
    for i in range(1, 101):
        x = np.loadtxt(f"{DEF_PATH}Set_{s}/{s}{i:03d}.txt")
        row = {"set": s, "id": i*10 + sets_index_map[s], "signal": x}
        rows.append(row)

df = pl.DataFrame(rows)


mapping1 = {
    "Z": "Normal", 
    "O": "Normal",
    "N": "interictal", 
    "F": "interictal",
    "S": "ictal",
}

mapping2 = {
    "Z": 0, 
    "O": 0,
    "N": 1, 
    "F": 1,
    "S": 2,
}

df = df.with_columns(pl.col("set").replace(mapping1).alias("classe"), pl.col("set").replace(mapping2).cast(pl.Int64).alias("classe_index"))

print(df.describe())
print(df.head(), df.tail())

shape: (9, 6)
┌────────────┬──────┬────────────┬────────┬────────────┬──────────────┐
│ statistic  ┆ set  ┆ id         ┆ signal ┆ classe     ┆ classe_index │
│ ---        ┆ ---  ┆ ---        ┆ ---    ┆ ---        ┆ ---          │
│ str        ┆ str  ┆ f64        ┆ f64    ┆ str        ┆ f64          │
╞════════════╪══════╪════════════╪════════╪════════════╪══════════════╡
│ count      ┆ 300  ┆ 300.0      ┆ 300.0  ┆ 300        ┆ 300.0        │
│ null_count ┆ 0    ┆ 0.0        ┆ 0.0    ┆ 0          ┆ 0.0          │
│ mean       ┆ null ┆ 508.0      ┆ null   ┆ null       ┆ 1.0          │
│ std        ┆ null ┆ 289.144164 ┆ null   ┆ null       ┆ 0.817861     │
│ min        ┆ N    ┆ 12.0       ┆ null   ┆ Normal     ┆ 0.0          │
│ 25%        ┆ null ┆ 262.0      ┆ null   ┆ null       ┆ 0.0          │
│ 50%        ┆ null ┆ 512.0      ┆ null   ┆ null       ┆ 1.0          │
│ 75%        ┆ null ┆ 754.0      ┆ null   ┆ null       ┆ 2.0          │
│ max        ┆ S    ┆ 1004.0     ┆ null   ┆ interi

In [3]:
X = np.stack(df["signal"].to_list()).astype(np.float32)
y = np.array(df["classe_index"].to_list(), dtype=np.int64)

# per-sample z-score (mean 0, std 1 across the 4097 samples)
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

In [4]:
ci_list = df["classe_index"].to_list()
has_nulls = any(v is None for v in ci_list)
ci = np.array([(-1 if v is None else v) for v in ci_list], dtype=np.int64)

print("has nulls:", has_nulls)
print("dtype:", df['classe_index'].dtype)
print("value counts:", np.unique(ci, return_counts=True))   # expect (0,1,2) with (200,200,100)
print("agreement with classe:", np.mean(ci == y))           # expect 1.0

has nulls: False
dtype: Int64
value counts: (array([0, 1, 2]), array([100, 100, 100]))
agreement with classe: 1.0


In [5]:
from train_kfold import main, plot_learning_curves, plot_confusion_matrix
results = main(X, y)

Device: cpu | samples: 300 | length: 4097 | classes: 3

Fold 1/10  (train=189, val=81, test=30)
      epoch   1/150  train_loss= 1.101 train_acc=0.333  val_loss= 1.100 val_acc=0.333
      epoch  25/150  train_loss= 1.099 train_acc=0.333  val_loss= 1.099 val_acc=0.333


KeyboardInterrupt: 

In [ ]:
plot_learning_curves(results)     # train/val loss + accuracy vs epoch
plot_confusion_matrix(results)    # aggregated 3×3 heatmap (Normal/Interictal/Ictal)